In [14]:
# DiT Block
# 图像生成领域不使用cross attention完成条件的引入是因为，这是一个O(n^2)的过程，对于生成图像太慢了
import torch
import torch.nn as nn
import math
from timm.models.vision_transformer import Attention, Mlp
from torch import Tensor

def modulate(x, shift: Tensor, scale: Tensor):
    return x * (1 + scale.unsqueeze(1) + shift.unsqueeze(1)) # unsqueeze插入一个维度

class DiTBlock(nn.Module):
    def __init__(self, hidden_size, num_heads, mlp_ratio=4.0): # Pointwise Feedforward里把mlp编到多少维度
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6) # 不需要自学习的隐射偏移参数
        self.attn = Attention(hidden_size, num_heads, qkv_bias=True)
        self.norm2 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        mlp_hidden_dim = int(hidden_size * mlp_ratio)
        approx_gelu = lambda: nn.GELU(approximate="tanh")
        self.mlp = Mlp(in_features=hidden_size, hidden_features= mlp_hidden_dim, act_layer=approx_gelu)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size * 6, bias=True),
        )

    def forward(self, x: Tensor, condition):
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = self.adaLN_modulation(condition).chunk(6, dim=1)
        x = x + self.attn(modulate(self.norm1(x), shift_msa, scale_msa)) * gate_msa.unsqueeze(1) # 三维乘三维

        x = x + self.mlp(modulate(self.norm2(x), shift_mlp, scale_mlp)) * gate_mlp.unsqueeze(1) # 三维乘三维
        return x
    

class FinalLayer(nn.Module):
    def __init__(self, hidden_size, patch_size, out_channels):
        super().__init__()
        self.norm_final = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.linear = nn.Linear(hidden_size, patch_size * patch_size * out_channels, bias=True)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size * 2, bias=True),
        )

    def forward(self, x: Tensor, condition):
        shift, scale = self.adaLN_modulation(condition).chunk(2, dim=1)
        x = modulate(self.norm_final(x), shift, scale) 
        x = self.linear(x)
        return x

In [15]:
def create_patches(image, patch_size=16):
    """
    将 256x256x3 的图像转换为 patch 序列
    返回: (B, num_patches, patch_size*patch_size*3)
    """
    B, C, H, W = image.shape
    assert H == W == 256, f"期望 256x256 的图像，但得到 {H}x{W}"
    assert H % patch_size == 0, f"图像尺寸必须能被 patch_size 整除"
    
    num_patches = (H // patch_size) ** 2
    
    # 使用 unfold 提取 patches
    patches = image.unfold(2, patch_size, patch_size).unfold(3, patch_size, patch_size)
    # patches shape: (B, C, num_patches_h, num_patches_w, patch_size, patch_size)
    patches = patches.permute(0, 2, 3, 1, 4, 5).contiguous()
    # patches shape: (B, num_patches_h, num_patches_w, C, patch_size, patch_size)
    patches = patches.view(B, num_patches, -1)
    # patches shape: (B, num_patches, C*patch_size*patch_size)
    
    return patches

def reconstruct_image(patches: Tensor, patch_size=16, image_size=256, out_channels=3):
    """
    将 patch 序列重构为图像
    输入: (B, num_patches, patch_size*patch_size*out_channels)
    输出: (B, out_channels, image_size, image_size)
    """
    B, num_patches, _ = patches.shape
    num_patches_h = image_size // patch_size
    assert num_patches == num_patches_h ** 2
    
    # 重塑为图像格式
    patches = patches.view(B, num_patches_h, num_patches_h, out_channels, patch_size, patch_size)
    patches = patches.permute(0, 3, 1, 4, 2, 5).contiguous()
    # (B, out_channels, num_patches_h, patch_size, num_patches_w, patch_size)
    image = patches.view(B, out_channels, image_size, image_size)
    
    return image

def verify_dit_process():
    print("=" * 60)
    print("DiT Block 验证 - 256x256x3 输入")
    print("=" * 60)
    
    # 超参数
    batch_size = 2
    hidden_size = 768
    num_heads = 12
    patch_size = 16
    out_channels = 3
    
    # 创建 dummy 数据
    print("\n1. 创建 dummy 图像: (2, 3, 256, 256)")
    dummy_image = torch.randn(batch_size, out_channels, 256, 256)
    print(f"   图像 shape: {dummy_image.shape}")
    print(f"   图像范围: [{dummy_image.min():.3f}, {dummy_image.max():.3f}]")
    
    # Patch embedding (简化版，实际使用线性投影)
    print(f"\n2. 转换为 patches (patch_size={patch_size})")
    patches = create_patches(dummy_image, patch_size)
    num_patches = patches.shape[1]
    print(f"   patches shape: {patches.shape}")
    print(f"   期望: ({batch_size}, {num_patches}, {patch_size*patch_size*out_channels})")
    print(f"   实际: ({batch_size}, {num_patches}, {patch_size*patch_size*out_channels})")
    
    # 简化的 patch embedding（将通道映射到 hidden_size）
    print(f"\n3. Patch embedding: {patch_size*patch_size*out_channels} -> {hidden_size}")
    patch_embed = nn.Linear(patch_size * patch_size * out_channels, hidden_size)
    x = patch_embed(patches)
    print(f"   嵌入后 shape: {x.shape}")
    print(f"   期望: ({batch_size}, {num_patches}, {hidden_size})")
    
    # 添加位置编码（简化）
    print(f"\n4. 添加位置编码")
    pos_embed = nn.Parameter(torch.zeros(1, num_patches, hidden_size))
    x = x + pos_embed
    print(f"   添加位置编码后 shape: {x.shape}")
    
    # 创建条件向量
    print(f"\n5. 创建条件向量 (timestep + class embedding)")
    condition = torch.randn(batch_size, hidden_size)
    print(f"   条件向量 shape: {condition.shape}")
    
    # DiT Block 前向传播
    print(f"\n6. DiT Block 前向传播")
    dit_block = DiTBlock(hidden_size, num_heads)
    
    print(f"   输入 x shape: {x.shape}")
    print(f"   输入 condition shape: {condition.shape}")
    print(f"   adaLN 调制输出: {hidden_size}*6 = {hidden_size*6} 维")
    print(f"   分割为: shift_msa({hidden_size}), scale_msa({hidden_size}), gate_msa({hidden_size}),")
    print(f"          shift_mlp({hidden_size}), scale_mlp({hidden_size}), gate_mlp({hidden_size})")
    
    x_out = dit_block(x, condition)
    print(f"   输出 shape: {x_out.shape}")
    print(f"   输入输出 shape 一致: {x.shape == x_out.shape}")
    
    # 统计参数量
    total_params = sum(p.numel() for p in dit_block.parameters())
    trainable_params = sum(p.numel() for p in dit_block.parameters() if p.requires_grad)
    print(f"\n   DiT Block 参数量:")
    print(f"   总参数: {total_params:,}")
    print(f"   可训练参数: {trainable_params:,}")
    
    # Final Layer 测试
    print(f"\n7. Final Layer 测试")
    final_layer = FinalLayer(hidden_size, patch_size, out_channels)
    
    print(f"   输入 x shape: {x_out.shape}")
    print(f"   输入 condition shape: {condition.shape}")
    print(f"   adaLN 调制输出: {hidden_size}*2 = {hidden_size*2} 维")
    print(f"   分割为: shift({hidden_size}), scale({hidden_size})")
    
    final_out = final_layer(x_out, condition)
    print(f"   输出 shape: {final_out.shape}")
    print(f"   期望: ({batch_size}, {num_patches}, {patch_size*patch_size*out_channels})")
    print(f"   期望值: ({batch_size}, {num_patches}, {16*16*3}) = ({batch_size}, {num_patches}, 768)")
    
    # 重构为图像
    print(f"\n8. 重构为图像")
    reconstructed = reconstruct_image(final_out, patch_size, 256, out_channels)
    print(f"   重构图像 shape: {reconstructed.shape}")
    print(f"   期望: ({batch_size}, {out_channels}, 256, 256)")
    print(f"   shape 匹配: {reconstructed.shape == dummy_image.shape}")
    
    # 验证数值稳定性
    print(f"\n9. 数值稳定性检查")
    print(f"   DiT Block 输出范围: [{x_out.min():.3f}, {x_out.max():.3f}]")
    print(f"   Final Layer 输出范围: [{final_out.min():.3f}, {final_out.max():.3f}]")
    print(f"   重构图像范围: [{reconstructed.min():.3f}, {reconstructed.max():.3f}]")
    print(f"   是否有 NaN: {torch.isnan(reconstructed).any()}")
    print(f"   是否有 Inf: {torch.isinf(reconstructed).any()}")
    
    # 内存使用估算
    print(f"\n10. 内存使用估算")
    x_memory = x.element_size() * x.nelement() / 1024**2
    attn_memory = x.element_size() * batch_size * num_heads * num_patches * num_patches / 1024**2
    print(f"   输入 x 内存: {x_memory:.2f} MB")
    print(f"   注意力矩阵内存: {attn_memory:.2f} MB")
    
    print("\n" + "=" * 60)
    print("验证完成！所有操作正常。")
    print("=" * 60)
    
    return True

if __name__ == "__main__":
    verify_dit_process()

DiT Block 验证 - 256x256x3 输入

1. 创建 dummy 图像: (2, 3, 256, 256)
   图像 shape: torch.Size([2, 3, 256, 256])
   图像范围: [-4.485, 4.855]

2. 转换为 patches (patch_size=16)
   patches shape: torch.Size([2, 256, 768])
   期望: (2, 256, 768)
   实际: (2, 256, 768)

3. Patch embedding: 768 -> 768
   嵌入后 shape: torch.Size([2, 256, 768])
   期望: (2, 256, 768)

4. 添加位置编码
   添加位置编码后 shape: torch.Size([2, 256, 768])

5. 创建条件向量 (timestep + class embedding)
   条件向量 shape: torch.Size([2, 768])

6. DiT Block 前向传播
   输入 x shape: torch.Size([2, 256, 768])
   输入 condition shape: torch.Size([2, 768])
   adaLN 调制输出: 768*6 = 4608 维
   分割为: shift_msa(768), scale_msa(768), gate_msa(768),
          shift_mlp(768), scale_mlp(768), gate_mlp(768)
   输出 shape: torch.Size([2, 256, 768])
   输入输出 shape 一致: True

   DiT Block 参数量:
   总参数: 10,628,352
   可训练参数: 10,628,352

7. Final Layer 测试
   输入 x shape: torch.Size([2, 256, 768])
   输入 condition shape: torch.Size([2, 768])
   adaLN 调制输出: 768*2 = 1536 维
   分割为: shift(768), scale(768

In [16]:
input = torch.randn(10, 3, 256, 256)

model = DiTBlock(hidden_size=768, num_heads=12)
condition = torch.randn(10, 768)
patches = create_patches(input, patch_size=16)
output = model(patches, condition=condition)    

print(output.shape)

torch.Size([10, 256, 768])
